# Entropy - Pythia Model Family

160M, 410M, 1B, 2.8B, 12B   

## Setup

In [ ]:
# Cell 0: Environment Detection
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

In [ ]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

In [ ]:
# Cell 2: Project Root & Path Setup
import sys

# Environment detection needed again after kernel restart
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}

In [3]:
from pathlib import Path

def find_project_root():
    path = Path.cwd()
    while path != path.parent:
        if (path / '.git').exists():
            return path
        path = path.parent
    raise FileNotFoundError("Couldn't find project root")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/trishasalas/Repos/Research/tmlr


In [ ]:
# Cell 3: Imports
import torch
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

In [ ]:
# Cell 5 - Model name variable
model_name = "pythia-12b"
short_name = model_name.split('/')[-1] 


In [ ]:
# Cell 6: Load Model
model = HookedTransformer.from_pretrained(f"EleutherAI/{model_name}")

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

### Entropy

In [ ]:
from pathlib import Path
import torch
import yaml
import pandas as pd
# Cell: Compute entropy (keep as a function — it's pure math)
def compute_entropy(logits):
    """H = -Σ P(x_i) log P(x_i)"""
    probs = torch.nn.functional.softmax(logits[0], dim=-1)
    log_probs = torch.log(probs + 1e-10)
    entropy = -torch.sum(probs * log_probs, dim=-1)
    return entropy

In [ ]:
# Cell: Run entropy analysis
prompts_path = PROJECT_ROOT / 'data' / 'all_prompts.yml'
with open(prompts_path, 'r') as f:
    templates = yaml.safe_load(f)

results = []

for case in templates['prompts']:
    prompt = case['prompt']
    tokens = model.to_tokens(prompt)
    logits = model(tokens)
    entropy = compute_entropy(logits)

    results.append({
        'prompt_id': case['prompt_id'],
        'concept': case['concept'],
        'prompt_type': case['prompt_type'],
        'template_type': case['template_type'],
        'prompt': prompt,
        'n_tokens': len(entropy),
        'mean_entropy': round(entropy.mean().item(), 4),
        'last_token_entropy': round(entropy[-1].item(), 4),
        'max_entropy': round(entropy.max().item(), 4),
        'min_entropy': round(entropy.min().item(), 4),
        'model': model_name
    })

entropy_df = pd.DataFrame(results)
entropy_df

In [ ]:
# Cell: Save results

output_dir = PROJECT_ROOT / 'results' / 'entropy' / suite
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'{short_name}-entropy.csv'
entropy_df.to_csv(output_path, index=False)
print(f"Saved {len(entropy_df)} entropy measurements to {output_path}")

In [ ]:
output_dir = PROJECT_ROOT / 'results' / 'entropy' / 'pythia' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / f'{model_name}-elicitation-entropy-binding.md', 'w') as f:
  f.write(f"# Model data captured during Elicitation/Entropy/Binding Battery\n")
  f.write(f"- Model name: {model_name}\n")
  f.write(f"- Model dtype: {next(model.parameters()).dtype}\n")
  f.write(f"- Layers: {model.cfg.n_layers}\n")
  f.write(f"- Heads: {model.cfg.n_heads}\n")
  f.write(f"- Hidden size: {model.cfg.d_model}\n")
  f.write(f"- Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M\n")



print(f"Saved to {output_dir}/{model_name}-model-dtype.md")

In [ ]:
import os
os.chdir(PROJECT_ROOT)
!git config user.email "trisha@trishasalas.com"
!git config user.name "Trisha Salas"
!git add results/
!git commit -m "entropy results: {model_name}"
!git push

### Delete Model & Clear Cache

In [ ]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")